# Bipartite graph scratch

In [229]:
import numpy as np
import polars as pl
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import optuna
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.svm import SVC, LinearSVC


In [2]:
rna = pl.read_csv("xena_brca_data/TCGA.BRCA.sampleMap_HiSeqV2", separator="\t")

In [3]:
# split into training and testing sets


sample,TCGA-AR-A5QQ-01,TCGA-D8-A1JA-01,TCGA-BH-A0BQ-01,TCGA-BH-A0BT-01,TCGA-A8-A06X-01,TCGA-A8-A096-01,TCGA-BH-A0C7-01,TCGA-AC-A5XU-01,TCGA-PE-A5DE-01,TCGA-PE-A5DC-01,TCGA-AR-A0TV-01,TCGA-GM-A3XG-01,TCGA-BH-A18J-01,TCGA-BH-A0W7-01,TCGA-E9-A3QA-01,TCGA-A7-A4SD-01,TCGA-BH-A0HA-01,TCGA-AR-A5QN-01,TCGA-A7-A0CH-11,TCGA-A7-A0CE-01,TCGA-AR-A0U1-01,TCGA-EW-A1OZ-01,TCGA-A2-A0EY-01,TCGA-A8-A09R-01,TCGA-LL-A440-01,TCGA-BH-A8FY-01,TCGA-E2-A1II-01,TCGA-A7-A6VX-01,TCGA-C8-A273-01,TCGA-BH-A1EO-01,TCGA-OL-A5RX-01,TCGA-BH-A0B9-01,TCGA-EW-A1P5-01,TCGA-AO-A03P-01,TCGA-AN-A0AS-01,TCGA-A2-A1G0-01,…,TCGA-BH-A5IZ-01,TCGA-A8-A09W-01,TCGA-E2-A107-01,TCGA-AR-A252-01,TCGA-C8-A12Z-01,TCGA-BH-A202-01,TCGA-AO-A1KT-01,TCGA-D8-A1XW-01,TCGA-D8-A1JU-01,TCGA-E9-A1N5-01,TCGA-A2-A4S1-01,TCGA-E2-A1IG-11,TCGA-E2-A153-01,TCGA-A2-A0YG-01,TCGA-BH-A0B7-01,TCGA-D8-A1X6-01,TCGA-BH-A0BL-01,TCGA-BH-A0DO-01,TCGA-A2-A04Q-01,TCGA-BH-A0B5-11,TCGA-BH-A1FE-06,TCGA-E9-A1NI-01,TCGA-BH-A0HY-01,TCGA-AR-A24T-01,TCGA-E9-A1NF-11,TCGA-AO-A0JA-01,TCGA-D8-A1XZ-01,TCGA-A7-A13E-11,TCGA-C8-A8HP-01,TCGA-E9-A5FL-01,TCGA-AC-A2FB-11,TCGA-E2-A15F-01,TCGA-A2-A3XT-01,TCGA-B6-A0X7-01,TCGA-BH-A1EV-11,TCGA-3C-AALJ-01,TCGA-B6-A0X1-01
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""ARHGEF10L""",9.5074,7.4346,9.3216,9.0198,9.6417,9.7665,10.0931,9.1524,9.9398,9.6287,9.6694,10.2287,9.2718,9.561,11.3338,11.3722,9.8078,10.1081,10.0819,9.5751,9.6685,9.0602,9.5451,9.6014,10.452,9.886,9.187,9.6146,9.7165,9.7915,10.1624,10.0971,9.7215,9.0184,8.7282,10.0161,…,11.6229,7.5288,10.3434,8.8947,8.8813,10.0359,8.6075,10.9642,9.7902,10.165,10.3455,10.0335,9.984,8.8155,9.1984,8.8809,9.5235,10.4995,10.064,9.3597,8.1078,9.7984,9.34,8.6545,10.4037,8.9976,8.5721,9.6265,10.1826,9.9199,9.909,10.0334,11.5144,10.5745,9.4048,10.9468,10.3164
"""HIF3A""",1.5787,3.6607,2.7224,1.3414,0.5819,0.2738,3.609,0.4738,2.9378,4.1136,0.433,5.342,2.1682,2.6648,2.5048,3.6372,1.9196,5.739,8.5517,4.1385,4.5824,0.7568,1.3008,0.656,6.4486,2.5831,5.4853,2.9753,2.1762,2.6206,3.7467,9.5132,0.4654,0.4157,0.0,6.2594,…,0.5021,0.8787,2.6529,4.4423,0.577,1.359,1.6344,2.3822,4.7678,2.154,2.1386,9.0708,1.6892,0.0,4.3177,0.5947,6.2779,3.9977,2.2799,8.8463,7.7224,4.227,0.0,0.4028,9.4111,0.7707,1.5668,8.1546,2.2159,3.8645,8.1872,0.8836,1.3169,4.0696,7.2537,0.931,2.4191
"""RNF17""",0.0,0.6245,0.5526,0.0,0.0,0.8765,0.0,0.0,0.0,0.0,0.0,0.0,0.4324,0.0,0.0,0.0,0.0,0.0,0.0,1.4303,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.9631,0.0,0.0,0.0,0.0,1.2109,…,0.0,0.0,0.0,0.3744,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.638,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.2703,0.0,0.0,0.0,3.7305,0.0,0.0,1.1329,0.4258,0.0,0.0,0.0
"""RNF10""",11.3676,11.9181,11.9665,13.1881,12.0036,11.8118,11.382,11.5004,12.2055,12.1312,11.9378,11.8971,12.3127,11.9861,11.1646,11.5724,12.4059,11.7531,11.9788,11.4626,12.4277,12.0018,11.5879,11.597,11.5529,12.0487,12.5989,11.3869,12.3921,11.5845,11.7273,12.0289,12.1964,12.2011,11.5278,11.7931,…,11.343,11.821,12.4371,11.6447,11.5965,11.4955,11.9798,11.4954,11.7845,11.8549,11.2387,11.8407,11.7244,12.2153,11.4783,12.1456,11.4204,11.8661,11.7065,12.1359,12.2541,12.5475,11.4961,11.6361,11.8686,12.0733,11.9794,11.9869,12.2653,12.4815,11.8263,12.0135,11.5818,11.8663,11.546,12.2616,12.157
"""RNF11""",11.1292,13.5273,11.4105,11.0911,11.2545,10.8554,10.7663,10.4358,11.221,10.8013,11.2889,11.3988,11.2739,10.841,10.8042,11.2059,11.4196,10.9013,11.5315,10.5215,10.5884,10.4395,11.1212,10.7365,10.9343,10.6782,10.8347,11.1685,10.7344,11.7243,11.236,9.6044,11.3045,11.2983,11.0501,11.8753,…,10.7503,11.4714,10.5106,11.4956,12.0406,11.2599,10.7881,11.6882,11.7708,11.1607,11.3168,11.7084,11.1408,11.3417,12.3537,11.1187,11.0695,11.1226,10.7574,12.3855,11.3686,10.6631,11.5667,11.1444,11.8553,11.1703,10.8409,11.9344,11.4117,10.4902,

In [7]:
gene_names = rna[:,0]
rna = rna.drop("sample")

In [13]:
# load labels

clinicalMatrix = pl.read_csv("xena_brca_data/TCGA.BRCA.sampleMap_BRCA_clinicalMatrix", separator="\t", infer_schema_length=0)

cM = clinicalMatrix.filter(pl.col("PAM50_mRNA_nature2012") != "null")
cM


sampleID,AJCC_Stage_nature2012,Age_at_Initial_Pathologic_Diagnosis_nature2012,CN_Clusters_nature2012,Converted_Stage_nature2012,Days_to_Date_of_Last_Contact_nature2012,Days_to_date_of_Death_nature2012,ER_Status_nature2012,Gender_nature2012,HER2_Final_Status_nature2012,Integrated_Clusters_no_exp__nature2012,Integrated_Clusters_unsup_exp__nature2012,Integrated_Clusters_with_PAM50__nature2012,Metastasis_Coded_nature2012,Metastasis_nature2012,Node_Coded_nature2012,Node_nature2012,OS_Time_nature2012,OS_event_nature2012,PAM50Call_RNAseq,PAM50_mRNA_nature2012,PR_Status_nature2012,RPPA_Clusters_nature2012,SigClust_Intrinsic_mRNA_nature2012,SigClust_Unsupervised_mRNA_nature2012,Survival_Data_Form_nature2012,Tumor_T1_Coded_nature2012,Tumor_nature2012,Vital_Status_nature2012,_INTEGRATION,_PANCAN_CNA_PANCAN_K8,_PANCAN_Cluster_Cluster_PANCAN,_PANCAN_DNAMethyl_BRCA,_PANCAN_DNAMethyl_PANCAN,_PANCAN_RPPA_PANCAN_K8,_PANCAN_UNC_RNAseq_PANCAN_K16,_PANCAN_miRNA_PANCAN,…,progesterone_receptor_level_cell_percent_category,project_code,radiation_therapy,sample_type,sample_type_id,surgical_procedure_purpose_other_text,system_version,targeted_molecular_therapy,tissue_prospective_collection_indicator,tissue_retrospective_collection_indicator,tissue_source_site,tumor_tissue_site,vial_number,vital_status,year_of_initial_pathologic_diagnosis,_GENOMIC_ID_TCGA_BRCA_exp_HiSeqV2_exon,_GENOMIC_ID_TCGA_BRCA_exp_HiSeqV2_PANCAN,_GENOMIC_ID_TCGA_BRCA_RPPA_RBN,_GENOMIC_ID_TCGA_BRCA_mutation,_GENOMIC_ID_TCGA_BRCA_PDMRNAseq,_GENOMIC_ID_TCGA_BRCA_hMethyl450,_GENOMIC_ID_TCGA_BRCA_RPPA,_GENOMIC_ID_TCGA_BRCA_PDMRNAseqCNV,_GENOMIC_ID_TCGA_BRCA_mutation_curated_wustl_gene,_GENOMIC_ID_TCGA_BRCA_hMethyl27,_GENOMIC_ID_TCGA_BRCA_PDMarrayCNV,_GENOMIC_ID_TCGA_BRCA_miRNA_HiSeq,_GENOMIC_ID_TCGA_BRCA_mutation_wustl_gene,_GENOMIC_ID_TCGA_BRCA_miRNA_GA,_GENOMIC_ID_TCGA_BRCA_exp_HiSeqV2_percentile,_GENOMIC_ID_data/public/TCGA/BRCA/miRNA_GA_gene,_GENOMIC_ID_TCGA_BRCA_gistic2thd,_GENOMIC_ID_data/public/TCGA/BRCA/miRNA_HiSeq_gene,_GENOMIC_ID_TCGA_BRCA_G4502A_07_3,_GENOMIC_ID_TCGA_BRCA_exp_HiSeqV2,_GENOMIC_ID_TCGA_BRCA_gistic2,_GENOMIC_ID_TCGA_BRCA_PDMarray
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""TCGA-A1-A0SD-0…","""Stage IIA""","""59""","""2""","""Stage IIA""","""437""",null,"""Positive""","""FEMALE""","""Negative""",null,null,null,"""Negative""","""M0""","""Negative""","""N0""","""437""","""0""","""LumA""","""Luminal A""","""Positive""",null,"""-9""","""-3""","""enrollment""","""T_Other""","""T2""","""LIVING""","""TCGA-A1-A0SD-0…","""High""","""C3-BRCA/Lumina…","""cluster 1""","""BRCA non-CIMP …",null,"""BRCA nonbasal-…","""miRNA cluster …",…,"""90-99%""",null,null,"""Primary Tumor""","""01""",null,"""6th""",null,"""NO""","""YES""","""A1""","""Breast""","""A""","""LIVING""","""2005""","""15bad71d-3031-…","""15bad71d-3031-…",null,"""TCGA-A1-A0SD-0…","""TCGA-A1-A0SD-0…",null,null,"""TCGA-A1-A0SD-0…","""TCGA-A1-A0SD-0…","""TCGA-A1-A0SD-0…","""TCGA-A1-A0SD-0…","""TCGA-A1-A0SD-0…","""TCGA-A1-A0SD-0…",null,"""15bad71d-3031-…",null,"""TCGA-A1-A0SD-0…","""TCGA-A1-A0SD-0…","""TCGA-A1-A0SD-0…","""15bad71d-3031-…","""TCGA-A1-A0SD-0…","""TCGA-A1-A0SD-0…"
"""TCGA-A1-A0SE-0…","""Stage I""","""56""","""2""","""Stage I""","""1320""",null,"""Positive""","""FEMALE""","""Negative""",null,null,null,"""Negative""","""M0""","""Negative""","""N0""","""1320""","""0""","""LumA""","""Luminal A""","""Positive""",null,"""-5""","""-7""","""enrollment""","""T1""","""T1""","""LIVING""","""TCGA-A1-A0SE-0…","""Iq""","""C3-BRCA/Lumina…","""cluster 1""","""BRCA non-CIMP …",null,"""BRCA nonbasal-…","""miRNA cluster …",…,"""90-99%""",null,null,"""Primary Tumor""","""01""",null,"""6th""",null,"""NO""","""YES""","""A1""","""Breast""",null,"""LIVING""","""2005""","""a998

In [17]:
labels = cM.select(["sampleID","PAM50_mRNA_nature2012"])
labels = labels.filter(pl.col("PAM50_mRNA_nature2012") != "Normal-like")

In [33]:
y_all = labels["PAM50_mRNA_nature2012"].to_numpy()
y_all

array(['Luminal A', 'Luminal A', 'Luminal A', 'Luminal A', 'Basal-like',
       'Luminal B', 'Basal-like', 'Luminal A', 'Basal-like', 'Basal-like',
       'Luminal B', 'Basal-like', 'Basal-like', 'Luminal A',
       'HER2-enriched', 'HER2-enriched', 'Luminal A', 'HER2-enriched',
       'Basal-like', 'Luminal A', 'Luminal A', 'Luminal A', 'Luminal B',
       'Luminal A', 'Luminal A', 'Luminal B', 'HER2-enriched',
       'HER2-enriched', 'Luminal A', 'Basal-like', 'HER2-enriched',
       'Basal-like', 'Luminal A', 'Luminal B', 'Luminal A', 'Luminal A',
       'Luminal A', 'HER2-enriched', 'Luminal B', 'Luminal A',
       'Luminal A', 'Luminal A', 'Luminal A', 'Luminal A', 'Luminal A',
       'Luminal B', 'Basal-like', 'Luminal A', 'Luminal B', 'Luminal B',
       'Basal-like', 'Luminal A', 'Basal-like', 'HER2-enriched',
       'Basal-like', 'Luminal B', 'Luminal B', 'Luminal A', 'Luminal A',
       'Luminal A', 'Luminal A', 'Luminal A', 'Basal-like', 'Luminal A',
       'Luminal B', 'Lum

In [18]:
np.unique(labels["PAM50_mRNA_nature2012"].to_numpy(), return_counts=True)

(array(['Basal-like', 'HER2-enriched', 'Luminal A', 'Luminal B'],
       dtype=object),
 array([ 98,  58, 231, 127]))

In [23]:
rna = rna.select(labels["sampleID"].to_numpy())

In [24]:
rna

TCGA-A1-A0SD-01,TCGA-A1-A0SE-01,TCGA-A1-A0SH-01,TCGA-A1-A0SJ-01,TCGA-A1-A0SK-01,TCGA-A1-A0SM-01,TCGA-A1-A0SO-01,TCGA-A2-A04N-01,TCGA-A2-A04P-01,TCGA-A2-A04Q-01,TCGA-A2-A04R-01,TCGA-A2-A04T-01,TCGA-A2-A04U-01,TCGA-A2-A04V-01,TCGA-A2-A04W-01,TCGA-A2-A04X-01,TCGA-A2-A04Y-01,TCGA-A2-A0CL-01,TCGA-A2-A0CM-01,TCGA-A2-A0CP-01,TCGA-A2-A0CQ-01,TCGA-A2-A0CS-01,TCGA-A2-A0CT-01,TCGA-A2-A0CU-01,TCGA-A2-A0CV-01,TCGA-A2-A0CW-01,TCGA-A2-A0CX-01,TCGA-A2-A0CY-01,TCGA-A2-A0CZ-01,TCGA-A2-A0D0-01,TCGA-A2-A0D1-01,TCGA-A2-A0D2-01,TCGA-A2-A0D3-01,TCGA-A2-A0D4-01,TCGA-A2-A0EM-01,TCGA-A2-A0EN-01,TCGA-A2-A0EO-01,…,TCGA-E2-A14W-01,TCGA-E2-A14X-01,TCGA-E2-A14Y-01,TCGA-E2-A14Z-01,TCGA-E2-A150-01,TCGA-E2-A152-01,TCGA-E2-A153-01,TCGA-E2-A154-01,TCGA-E2-A155-01,TCGA-E2-A156-01,TCGA-E2-A158-01,TCGA-E2-A159-01,TCGA-E2-A15A-01,TCGA-E2-A15C-01,TCGA-E2-A15D-01,TCGA-E2-A15E-01,TCGA-E2-A15F-01,TCGA-E2-A15G-01,TCGA-E2-A15H-01,TCGA-E2-A15I-01,TCGA-E2-A15J-01,TCGA-E2-A15K-01,TCGA-E2-A15L-01,TCGA-E2-A15M-01,TCGA-E2-A15O-01,TCGA-E2-A15P-01,TCGA-E2-A15R-01,TCGA-E2-A15S-01,TCGA-E2-A15T-01,TCGA-E2-A1AZ-01,TCGA-E2-A1B0-01,TCGA-E2-A1B1-01,TCGA-E2-A1B4-01,TCGA-E2-A1B5-01,TCGA-E2-A1B6-01,TCGA-E2-A1BC-01,TCGA-E2-A1BD-01
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
9.1657,9.7179,8.8669,9.7663,9.8632,8.8234,8.6701,9.9831,10.5102,10.064,9.5425,10.449,9.4763,10.5333,8.9774,8.9665,9.3774,9.5693,11.4132,9.6413,10.3112,9.2703,9.0876,11.2247,8.9437,9.741,8.9323,8.4062,10.1639,8.3209,11.1395,10.1611,9.2467,10.9353,9.3572,10.0545,9.778,…,8.9611,10.7636,9.9899,9.8046,11.2344,9.8236,9.984,9.2405,9.8415,10.3731,9.2199,10.1219,9.1705,10.0851,9.2262,9.7065,10.0334,10.0949,9.4848,9.1406,9.2531,9.9412,9.3713,8.453,9.376,9.1228,9.1934,10.0346,9.8209,9.1036,8.5746,9.3223,10.4367,8.9027,8.6965,9.0303,10.7991
2.4935,3.3983,1.6631,6.1765,7.497,1.8177,8.9699,1.7558,6.9166,2.2799,1.4911,9.2573,6.3214,2.9105,2.5516,1.6612,3.6294,3.6849,0.0,4.8982,0.4407,2.3655,1.1726,2.9769,3.0291,3.35,0.8538,5.0676,4.063,1.7954,1.9355,2.1132,0.0,4.0158,1.4728,4.2849,2.5901,…,1.7638,4.8435,2.8783,2.9498,2.9782,2.3097,1.6892,1.9935,2.1329,0.7806,4.3358,4.1235,0.7135,0.848,3.8032,4.4089,0.8836,2.4616,1.1422,3.0117,0.4366,1.3397,1.6929,4.9786,0.8672,1.7922,0.3056,5.0409,0.3936,5.7418,1.3134,4.919,1.8654,4.8873,4.6748,6.9578,2.1772
0.0,0.0,0.4449,0.0,0.7051,1.178,0.0,0.0,0.6028,0.0,0.0,0.0,0.0,2.0906,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.4891,0.6906,0.0,7.7299,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.8064,0.0,0.0,0.0,0.0,0.638,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.4333,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0372,0.0,0.0,1.4776,0.0,0.8114
11.7868,11.3219,11.8779,11.502,12.099,11.9934,11.6209,12.185,12.1741,11.7065,11.9362,11.3839,12.5415,12.22,12.0188,12.5742,12.0841,11.6492,12.0196,12.0527,12.2531,12.3055,11.1702,11.5801,12.0265,12.2589,11.945,12.9661,11.6858,11.8358,13.0469,11.7749,11.9473,12.1437,11.6602,12.0735,11.881,…,12.3585,11.8525,11.7697,12.3889,11.3972,12.1153,11.7244,11.5464,11.9198,12.0953,11.8904,12.0105,12.2024,11.9198,11.9175,11.7502,12.0135,12.2843,12.3655,12.0196,11.9345,11.9704,11.88,11.7094,12.4611,12.4242,11.7385,12.4475,11.8324,11.416,11.7854,11.7005,12.428,12.6904,12.3403,11.8271,12.2783
11.0131,11.2617,11.8526,11.4405,9.9933,11.9332,11.0278,11.3242,11.2632,10.7574,10.4421,10.6897,10.9603,11.2698,11.9258,11.0061,10.9864,11.0977,11.1327,11.1117,11.1687,10.7255,11.4965,11.116,11.5781,10.6494,10.7706,10.4751,11.2883,10.838,11.5712,10.9105,12.0452,10.1117,11.7259,10.9669,11.3562,…,11.3929,11.1539,11.0064,10.2678,11.758,11.8669,11.1408,10.3065,11.696,11.6226,10.6569,12.1943,10.6067,11.4468,11.066,11.5324,10.837,11.112,10.7207,11.3718,10.688,10.898,11.183,11.415,10.718,10.5873,10.1383,10.8017,11.1716,11.2437,10.9101,11.6778,11.0804,11.

In [48]:
# labels and rna columns are aligned, do train, val, test split

# random split for train / test
train_temp_idx, test_idx = train_test_split(
    np.arange(rna.shape[1]),
    test_size=0.3,
    # stratify=y,
    random_state=4
)

In [57]:
train_idx, val_idx = train_test_split(
    train_temp_idx, 
    test_size=0.1, 
    stratify=y_all_e[train_temp_idx],                   
    random_state=4
)

In [34]:
label_encoder = LabelEncoder()
label_encoder.fit(y_all)
y_all_e = label_encoder.transform(y_all)

# add numberd labels to labels df
labels = labels.with_columns(y=y_all_e)

In [138]:
# split dataframes
train_df = rna[:, train_idx.tolist()]
train_labels = labels[train_idx]
val_df = rna[:, val_idx.tolist()]
val_labels = labels[val_idx]
test_df = rna[:, test_idx.tolist()]
test_labels = labels[test_idx]

In [147]:
# select features based on the training set
genes_class_mean = np.zeros((rna.shape[0], 4))

# separate training samples by class

for label in range(4):
    # select cols which are from this class
    label_samples = train_labels.filter(pl.col("y") == label)["sampleID"].to_numpy()
    label_train_df = train_df[label_samples].to_numpy()

    genes_class_mean[:, label] = label_train_df.mean(axis=1)

In [213]:
genes_class_variance = genes_class_mean.var(axis=1)

class_var_order = np.argsort(genes_class_variance)

# select the 500 genes with the largest variance
best_500 = class_var_order >= (genes_class_variance.shape[0] - 500)
best_500_idx = np.where(best_500)[0]

# selection itself
train_df = train_df[best_500_idx]
val_df = val_df[best_500_idx]
test_df = test_df[best_500_idx]

In [256]:
def min_max_scale(X):
    return (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))

In [259]:
X_train = train_df.to_numpy().T
y_train = train_labels["y"].to_numpy()
X_val = val_df.to_numpy().T
y_val = val_labels["y"].to_numpy()
X_test = test_df.to_numpy().T
y_test = test_labels["y"].to_numpy()

# scale everything to [0,1]
# X_train = min_max_scale(X_train)
# X_val = min_max_scale(X_val)
# X_test = min_max_scale(X_test)

In [260]:
# Example data
data = np.array(
    [[1, 2, 3],
    [4, 5, 6],
    [7, 8, 9]]
)

# Fit the scaler to the data and transform the data
scaled_data = (data - data.min(axis=0)) / (data.max(axis=0) - data.min(axis=0))

print("Original data:")
print(data)
print("\nScaled data:")
print(scaled_data)


Original data:
[[1 2 3]
 [4 5 6]
 [7 8 9]]

Scaled data:
[[0.  0.  0. ]
 [0.5 0.5 0.5]
 [1.  1.  1. ]]


In [261]:
C_range: tuple = (1e-3, 1e3)
gamma_range: tuple = (1e-3, 1e3)

# eval on rbf svms
def objective(trial):

    svm = SVC(
        C=trial.suggest_float("C", C_range[0], C_range[1], log=True),
        gamma=trial.suggest_float(
            "gamma", gamma_range[0], gamma_range[1], log=True
        ),
        class_weight=trial.suggest_categorical("class_weight", ["balanced", None]),
        kernel="rbf",
    )

    svm.fit(X_train, y_train)
    
    y_pred = svm.predict(X_val)

    f1 = f1_score(y_val, y_pred, average="weighted")

    return f1

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=200)

# print the mean f1 score for the best performing parameter
# print(
#     f"| RBF SVM | {best_results['acc']:.4f} +/- {best_results['acc_std']:.4f} | {best_results['f1_macro']:.4f} +/- {best_results['f1_macro_std']:.4f} | {best_results['f1_weighted']:.4f} +/- {best_results['f1_weighted_std']:.4f} |"
# )
print(f"study {study.best_value=}, {study.best_params=}")

[I 2024-03-14 02:06:08,381] A new study created in memory with name: no-name-bd4c351b-9478-4d09-b936-35bc0dca5928
[I 2024-03-14 02:06:08,418] Trial 0 finished with value: 0.2450980392156863 and parameters: {'C': 0.10927203746599043, 'gamma': 0.007756868910293816, 'class_weight': None}. Best is trial 0 with value: 0.2450980392156863.
[I 2024-03-14 02:06:08,444] Trial 1 finished with value: 0.2450980392156863 and parameters: {'C': 30.560501464987727, 'gamma': 0.3212980953801838, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.2450980392156863.
[I 2024-03-14 02:06:08,480] Trial 2 finished with value: 0.2450980392156863 and parameters: {'C': 25.157108697794, 'gamma': 285.62341300943655, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.2450980392156863.
[I 2024-03-14 02:06:08,504] Trial 3 finished with value: 0.2450980392156863 and parameters: {'C': 0.013582147420590333, 'gamma': 0.11336990558388652, 'class_weight': None}. Best is trial 0 with value: 0.2450980392156863.

study study.best_value=0.9128808611567233, study.best_params={'C': 0.13263430174534643, 'gamma': 0.0010014274339660567, 'class_weight': 'balanced'}


In [268]:
# fit model with best params and test on test set
svm = SVC(
        C=0.13263430174534643,
        gamma=0.0010014274339660567,
        class_weight="balanced",
        kernel="rbf",
)

svm.fit(X_train, y_train)

y_pred = svm.predict(X_test)

f1 = f1_score(y_test, y_pred, average="weighted")

print(f1)

0.795370962228466


## SVMs with rbf kernel
- weighted f1 score with ~0.8 on test set

In [ ]:
# build mogonet style GNN with
# a. thresholded cosine similarity


In [269]:
def cosine_similarity_matrix(matrix):
    # Compute dot product between all pairs of vectors
    dot_products = np.dot(matrix, matrix.T)
    
    # Compute magnitudes of all vectors
    magnitudes = np.linalg.norm(matrix, axis=1)
    
    # Compute outer product of magnitudes to obtain matrix of magnitudes product
    magnitude_products = np.outer(magnitudes, magnitudes)
    
    # Compute cosine similarity matrix
    cosine_similarities = dot_products / magnitude_products
    
    return cosine_similarities

# Example matrix of vectors (rows represent vectors)
matrix = np.array([[1, 1, 1],
                   [2, 2, 2],
                   [1, -1, 1]])

# Compute cosine similarity between all vectors
cos_sim_matrix = cosine_similarity_matrix(matrix)

print("Cosine similarity matrix:")
print(cos_sim_matrix)

Cosine similarity matrix:
[[1.         1.         0.33333333]
 [1.         1.         0.33333333]
 [0.33333333 0.33333333 1.        ]]


In [ ]:
cosine_similarity_matrix()

In [ ]:
# b. moglam style cosine similarity + learned graph

In [ ]:
# build bipartite graph style GNN

In [ ]:
# a. w interactions

In [ ]:
# b. w/o interactions

In [ ]:
# eval mogonet style model


In [ ]:
# eval bipartite style model
